# LLM章の入口

この notebook は、LLM 章を読み始める前に『この章では結局何ができるようになればよいのか』をはっきりさせるための導入です。LLM は難しい記号の章ではなく、文章を入力すると続きを予測する形で学んだモデルを、どう賢く安全に使うかを学ぶ章です。話題が広く見えやすいので、最初に地図を作ります。

## LLM は『文章を扱う機械学習』から始まる

LLM は巨大なモデルですが、土台はこれまで見てきた機械学習やディープラーニングと同じです。入力を受け取り、予測を出し、間違いを減らすように学習し、実際の場面で使います。違うのは、入力も出力も文章になりやすく、人に見せる答えをそのまま作るので、使い方の設計が特に重要になる点です。

たとえばチャットAIは、見た目は会話しているようでも、中では『ここまでの文章の続きとして、次にどんな単語が来そうか』を何度も予測しています。この章でも、毎回『何を入力しているか』『学習では何を良しとしているか』『モデルの外に任せるべき仕事は何か』を追えば、話題が変わっても軸を保てます。

## 最初に知っておきたいこと

- LLM は、会話を理解している魔法の箱ではなく、文章の続きを予測するように学んだモデルです
- だから、質問の書き方を変えるだけでも答えがかなり変わります
- ただし、モデルの中に入っていない最新情報や個別の社内情報は、自力では知りません
- そのため、学習そのものだけでなく、検索やツールとどう組み合わせるかも重要になります

この 4 点を先に持っておくと、LLM を過大評価しすぎたり、逆に『結局あてにならない』と切り捨てたりしにくくなります。

## 章全体の見取り図

1. プロンプトエンジニアリング: モデルへどう頼めば答えが変わるかを見る
2. 事前学習: なぜ『続きを予測する練習』で文章らしい出力ができるのかを見る
3. スケーリング則: データ量・計算量・モデルサイズを増やすと何が変わるか整理する
4. ファインチューニング: 特定の目的に合わせて追加学習する意味を理解する
5. ハルシネーションと RLHF: もっともらしい誤答をどう減らし、人にとって望ましい振る舞いへ寄せるかを見る
6. Tool Use と RAG: モデル単体では足りない情報を、検索やツールでどう補うか学ぶ
7. 軽量化: 速さやコストの制約がある中で、何を残して何を削るか考える

つまりこの章は、ただ新しい手法名を並べる章ではありません。『どう頼むか』『どう学ばせるか』『どう安全に使うか』の 3 つの観点で、LLM を分けて理解する章だと考えると整理しやすくなります。

## 読み進めるときの3つの見方

- 入力を工夫する: 指示文、会話履歴、検索結果をどう渡すか
- 学習で身につける: 何を正解として追加学習し、どんな癖を覚えさせるか
- 運用で失敗を減らす: 使ってはいけない出力をどう抑え、遅さやコストをどう管理するか

この 3 つを意識すると、似た『改善』の話を分けて考えられます。たとえば、プロンプトを直すのは入力の工夫であり、ファインチューニングは学習内容の工夫です。さらに、危ない出力を止めるルール作りは運用の工夫です。

## 最初に揃えたい前提

- ディープラーニング章で見た『入力から予測を作り、間違いを減らすように学習する』という流れ
- Python で簡単な文字列処理やリスト処理を追えること
- 分からない単語が出たら、その場で小さく意味を確かめながら読めること

最初から `token` `embedding` `attention` などの語彙を知っている必要はありません。この章の前提は、巨大な学習基盤の知識ではなく、『文章を扱うと何が難しくなるのか』を順に追う姿勢です。専門用語は必要になった地点で覚えれば十分です。

In [ ]:
question = '東京の今日の気温を答えて'
needs_external = any(key in question for key in ['今日', '最新', '気温'])
route = 'retrieval_or_tool' if needs_external else 'model_only'
print('route =', route)

この短い例だけでも、LLM では『文章を生成できる』だけでは足りないことが見えます。質問の中に `今日` や `最新` が入っているなら、モデルの記憶だけに頼らず、外の情報を取りに行くべきかを考える必要があります。後半の RAG や Tool Use は、この判断をきちんと仕組みにする話です。

## よくある混乱

- 指示の出し方を直す話と、モデル自体を追加学習する話が混ざる
- 学習で直すべき問題と、運用時のルールで止めるべき問題が混ざる
- RAG で情報を取ってきただけで、正しい答えになった気になる
- モデルを軽くした結果、答えの質がどこまで落ちたかを見なくなる

この章では、同じ『改善』でもどこへ手を入れているかを毎回切り分けます。問題の場所を取り違えると、効くはずのない対策を選んでしまいます。

In [ ]:
layers = {
    'prompt': '入力を変える',
    'fine_tuning': '重みを変える',
    'rails': '推論時の入出力を制御する',
}
for name, role in layers.items():
    print(name, '->', role)

この 3 つはどれも品質改善に見えますが、直している場所が違います。まず『頼み方を直す話か』『学習内容を直す話か』『運用ルールを直す話か』を分けることが、LLM を落ち着いて理解する第一歩です。

## この章を読み終えたときの目標

この章の到達点は、『LLM はすごいから何でもできる』という見方をやめて、どこまでがモデル本体の力で、どこからが入力設計や追加学習や運用設計の仕事なのかを説明できるようになることです。そこまで行けば、新しい手法名が出てきても慌てずに『これは何を改善するための技術か』と整理できます。